# 실험 01: Agent의 무한 실행 루프 (AgentExecutor)

## 1. 실험 제목과 목표
- **제목**: 스스로 생각하고 행동하는 시스템
- **목표**: 교안 13번 슬라이드에 명시된 "목표 이해 -> Tool 선택 -> 실행 -> 결과 확인 -> 다음 행동 판단"의 **Execution Loop(실행 루프)**를 LangChain의 `AgentExecutor`를 통해 자동화합니다.

## 2. 실험 개요
1. 지난 챕터(05_tool_calling)에서는 우리가 `for` 문을 돌려 툴을 수동으로 실행해주었습니다.
2. 이번에는 LLM이 **목표 달성할 때까지** 알아서 툴을 여러 번 호출하는 진정한 Agent를 만들어 봅니다.
3. **관전 포인트**: "2024년 파리 올림픽 개최국이 어디야? 그리고 그 국가의 영문 이름 글자 수에 10을 곱하면 얼마야?" 처럼 검색과 계산을 연계해야 풀 수 있는 복합 질문 던지기.

## 3. 라이브러리 import

In [6]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

## 4. 환경 변 로드 및 Tool 세팅

In [7]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

# 1. 계산기 툴 정의
@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다. 파이썬의 eval()을 사용하여 계산 가능한 수식을 문자열로 입력하세요."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"계산 에러: {e}"

# 2. 웹 검색 툴 정의
search_tool = DuckDuckGoSearchRun(
    name="web_search",
    description="최신 인터넷 정보가 필요할 때 사용합니다."
)

tools = [calculator, search_tool]

## 5. Agent 설계 및 실행 (Execution Loop)
Agent를 만들기 위해서는 **1) 프롬프트(지시사항)**, **2) LLM**, **3) Tools** 세 가지가 필요합니다.

In [8]:
# Agent를 위한 기본 프롬프트 (교안의 Instructions/System Prompt 역할)
# 반드시 {agent_scratchpad} 변수가 필요합니다. (여기에 Tool 실행 결과들이 차곡차곡 누적됩니다)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 만능 비서입니다. 질문에 답하기 위해 필요한 도구를 자유롭게 사용하세요."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 1. Agent 생성 (LLM의 '두뇌' 역할. 어떤 툴을 쓸지 판단만 함)
agent = create_tool_calling_agent(llm, tools, prompt)

# 2. Agent Executor 생성 (Agent의 '손발' 역할. 실제로 툴을 실행하고 무한 루프를 도는 엔진)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True) # verbose=True로 설정하면 에이전트의 생각 과정을 볼 수 있습니다.

In [9]:
print("=== [다단계 추론 테스트 시작] ===")
# 검색(국가 찾기) -> 문자열 길이 계산(어? 이건 툴이 없는데 머리로 해야지) -> 계산기 툴(곱하기) 의 복합 과정이 필요함
question = "2024년 하계 올림픽 개최국의 영어 이름 글자 수를 세고, 거기에 15를 곱하면 얼마야?"

# .invoke() 하나만 호출하면 알아서 끝까지 해결하고 옴
result = agent_executor.invoke({"input": question})

print("\n[최종 답변]")
print(result["output"])

=== [다단계 추론 테스트 시작] ===


> Entering new AgentExecutor chain...

Invoking: `web_search` with `{'query': '2024 Summer Olympics host country'}`


2 weeks ago - In France, domestic rights to the 2024 Summer Olympics were owned by Warner Bros. Discovery (formerly Discovery Inc.) via Eurosport, with free-to-air coverage sublicensed to the country's public broadcaster France Télévisions. 1 week ago - In 2022, Beijing became the first city to have hosted both the Summer and Winter Olympics. By 2034, eleven cities will have hosted the Olympic Games more than once: Athens (1896 and 2004 Summer Olympics), Paris (1900, 1924 and 2024 Summer Olympics), London (1908, 1948 and 2012 Summer Olympics), St. May 12, 2026 - This is a chronological summary of the major events of the 2024 Summer Olympics in Paris and other venues in Metropolitan France, plus one subsite in Tahiti in the overseas country of French Polynesia. Competition began on 24 July with the first matches in the group stages of football a

## 6. 결과 해석

위의 초록색 로그(`verbose=True`)를 유심히 살펴보세요.
1. **첫 번째 판단**: "어? 2024 올림픽 개최국을 알아야겠네?" -> `web_search` 툴 호출
2. **실행 결과 확인**: "아하, France(프랑스)구나."
3. **두 번째 판단**: "France는 6글자네. 이제 6에 15를 곱해야지." -> `calculator` 툴에 `6 * 15` 전송
4. **실행 결과 확인**: "90이네."
5. **최종 답변 생성**: "모든 목표를 달성했다! 최종 답변을 정리해서 주자."

이것이 바로 교안에 설명된 **Execution Loop(실행 루프)**의 정수입니다.

## 7. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 05번 챕터에서는 사용자가 개입해서 툴을 실행해줬지만, `AgentExecutor`를 쓰니 마치 자아를 가진 봇처럼 스스로 목표를 쪼개서 차근차근 해결함.
* 하지만 현재 에이전트는 "건망증"이 있음. 첫 번째 질문을 던지고, 두 번째 질문에서 "아까 그거 다시 말해봐"라고 하면 기억을 못함.

### 다음 개선 방향
* 교안의 구성 요소 중 **Memory(이전 대화와 작업 상태 저장)** 기능을 Agent에 부착하여, 과거의 맥락을 기억하며 행동하는 에이전트를 만들어야 함.